In [1]:
print("Hello spark hell yeahhh")

Hello spark hell yeahhh


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
        .appName("ECommerceBigData")\
            .master("local[*]")\
                .getOrCreate()
                

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/20 17:22:25 WARN Utils: Your hostname, pranav-ASUS-TUF-Gaming-A15-FA506ICB-FA506ICB, resolves to a loopback address: 127.0.1.1; using 192.168.0.157 instead (on interface wlp3s0)
26/09/20 17:22:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 17:22:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.version

'4.2.0'

In [5]:
spark


In [6]:
data = [
    ("1001", "C001", "Laptop", 1, 55000),
    ("1002", "C002", "Mouse", 2, 800),
    ("1003", "C001", "Keyboard", 1, 1500),
    ("1004", "C003", "Monitor", 1, 12000),
    ("1005", "C002", "Headphones", 1, 2500)
]

In [7]:
orders = spark.createDataFrame(
    data,
    [
        "order_id",
        "customer_id",
        "product",
        "quantity",
        "price"
    ]
)

In [8]:
orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
+--------+-----------+----------+--------+-----+



In [9]:
orders.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- price: long (nullable = true)



In [10]:
orders.select(
    "customer_id",
    "product",
    "price"
).show()

+-----------+----------+-----+
|customer_id|   product|price|
+-----------+----------+-----+
|       C001|    Laptop|55000|
|       C002|     Mouse|  800|
|       C001|  Keyboard| 1500|
|       C003|   Monitor|12000|
|       C002|Headphones| 2500|
+-----------+----------+-----+



In [12]:
orders.filter(
    orders.price >10000
).show()

+--------+-----------+-------+--------+-----+
|order_id|customer_id|product|quantity|price|
+--------+-----------+-------+--------+-----+
|    1001|       C001| Laptop|       1|55000|
|    1004|       C003|Monitor|       1|12000|
+--------+-----------+-------+--------+-----+



In [13]:
orders= orders.withColumn(
    "total_amount",
    orders.quantity * orders.price
)

In [14]:
orders.show()

+--------+-----------+----------+--------+-----+------------+
|order_id|customer_id|   product|quantity|price|total_amount|
+--------+-----------+----------+--------+-----+------------+
|    1001|       C001|    Laptop|       1|55000|       55000|
|    1002|       C002|     Mouse|       2|  800|        1600|
|    1003|       C001|  Keyboard|       1| 1500|        1500|
|    1004|       C003|   Monitor|       1|12000|       12000|
|    1005|       C002|Headphones|       1| 2500|        2500|
+--------+-----------+----------+--------+-----+------------+



In [15]:
orders.groupBy("customer_id").sum("total_amount").show()

+-----------+-----------------+
|customer_id|sum(total_amount)|
+-----------+-----------------+
|       C001|            56500|
|       C002|             4100|
|       C003|            12000|
+-----------+-----------------+



In [16]:
from pyspark.sql.functions import sum

In [17]:
customer_spending = orders.groupBy(
    "customer_id"
).agg(
    sum ("total_amount").alias("total_spending")
)

In [18]:
customer_spending.show()

+-----------+--------------+
|customer_id|total_spending|
+-----------+--------------+
|       C001|         56500|
|       C002|          4100|
|       C003|         12000|
+-----------+--------------+



In [19]:
customer_spending.orderBy(
    "total_spending",
    ascending = False
).show()

+-----------+--------------+
|customer_id|total_spending|
+-----------+--------------+
|       C001|         56500|
|       C003|         12000|
|       C002|          4100|
+-----------+--------------+



In [21]:
orders.filter(
    orders.quantity >1 
).show(
)

+--------+-----------+-------+--------+-----+------------+
|order_id|customer_id|product|quantity|price|total_amount|
+--------+-----------+-------+--------+-----+------------+
|    1002|       C002|  Mouse|       2|  800|        1600|
+--------+-----------+-------+--------+-----+------------+



In [24]:
orders.orderBy(
    "total_amount",
    ascending = False
).show()

+--------+-----------+----------+--------+-----+------------+
|order_id|customer_id|   product|quantity|price|total_amount|
+--------+-----------+----------+--------+-----+------------+
|    1001|       C001|    Laptop|       1|55000|       55000|
|    1004|       C003|   Monitor|       1|12000|       12000|
|    1005|       C002|Headphones|       1| 2500|        2500|
|    1002|       C002|     Mouse|       2|  800|        1600|
|    1003|       C001|  Keyboard|       1| 1500|        1500|
+--------+-----------+----------+--------+-----+------------+



In [25]:
from pyspark.sql.functions import avg

In [27]:
orders.select(
    avg("price").alias("average_price")
).show()

+-------------+
|average_price|
+-------------+
|      14360.0|
+-------------+



In [28]:
from pyspark.sql.functions import count

In [29]:
orders.groupBy(
    "customer_id"
).agg(
    count("*").alias("number_of_orders")
).show()

+-----------+----------------+
|customer_id|number_of_orders|
+-----------+----------------+
|       C001|               2|
|       C002|               2|
|       C003|               1|
+-----------+----------------+



In [30]:
orders.select(
    sum("total_amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|        72600|
+-------------+



In [ ]:
##part 5 reading data form hdfs


In [31]:
hdfs_orders = spark.read.parquet(
    "hdfs://localhost:9000/ecommerce/processed/orders"
)

In [32]:
hdfs_orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
+--------+-----------+----------+--------+-----+



In [33]:
hdfs_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



In [34]:
hdfs_orders = hdfs_orders.withColumn(
    "total_amount",
    hdfs_orders.quantity * hdfs_orders.price
)

In [35]:
hdfs_orders.groupBy(
    "customer_id"
).agg(
    sum("total_amount").alias("total_spending")
).orderBy(
    "total_spending",
    ascending=False
).show()

+-----------+--------------+
|customer_id|total_spending|
+-----------+--------------+
|       C001|         56500|
|       C003|         12000|
|       C002|          4100|
+-----------+--------------+



In [36]:
## part 6 reading raw csv data

In [39]:
raw_orders = spark.read.csv(
    "hdfs://localhost:9000/ecommerce/raw/orders.csv",
    header=True,
    inferSchema = True
)

In [40]:
raw_orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
+--------+-----------+----------+--------+-----+



In [41]:
raw_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



In [42]:
dirty_orders = spark.read.csv(
    "hdfs://localhost:9000/ecommerce/raw/orders_dirty.csv",
    header= True,
    inferSchema= True
)

In [43]:
dirty_orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
|    1006|       NULL|    Webcam|       1| 2000|
|    1007|       C004|  Keyboard|      -2| 1500|
|    1008|       C005|     Mouse|       1| -500|
|    1008|       C005|     Mouse|       1| -500|
+--------+-----------+----------+--------+-----+



In [44]:
dirty_orders.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



In [45]:
print("Rows:",dirty_orders.count())

Rows: 9


In [46]:
dirty_orders.show(truncate=False)

+--------+-----------+----------+--------+-----+
|order_id|customer_id|product   |quantity|price|
+--------+-----------+----------+--------+-----+
|1001    |C001       |Laptop    |1       |55000|
|1002    |C002       |Mouse     |2       |800  |
|1003    |C001       |Keyboard  |1       |1500 |
|1004    |C003       |Monitor   |1       |12000|
|1005    |C002       |Headphones|1       |2500 |
|1006    |NULL       |Webcam    |1       |2000 |
|1007    |C004       |Keyboard  |-2      |1500 |
|1008    |C005       |Mouse     |1       |-500 |
|1008    |C005       |Mouse     |1       |-500 |
+--------+-----------+----------+--------+-----+



In [58]:
clean_orders= dirty_orders

In [59]:
clean_orders= clean_orders.dropna()

In [60]:
clean_orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
|    1007|       C004|  Keyboard|      -2| 1500|
|    1008|       C005|     Mouse|       1| -500|
|    1008|       C005|     Mouse|       1| -500|
+--------+-----------+----------+--------+-----+



In [61]:
clean_orders = clean_orders.filter(
    clean_orders.quantity> 0,
    
)

In [62]:
clean_orders = clean_orders.filter(
    clean_orders.price> 0
)

In [63]:
clean_orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
+--------+-----------+----------+--------+-----+



In [64]:
clean_orders = clean_orders.dropDuplicates(["order_id"])

In [66]:
clean_orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
+--------+-----------+----------+--------+-----+



In [67]:
print("clean rows ", clean_orders.count())

clean rows  5


In [68]:
clean_orders.filter(
    clean_orders.quantity <= 0
).show()

+--------+-----------+-------+--------+-----+
|order_id|customer_id|product|quantity|price|
+--------+-----------+-------+--------+-----+
+--------+-----------+-------+--------+-----+



In [69]:
clean_orders.filter(
    clean_orders.price <= 0
).show()

+--------+-----------+-------+--------+-----+
|order_id|customer_id|product|quantity|price|
+--------+-----------+-------+--------+-----+
+--------+-----------+-------+--------+-----+



In [70]:
clean_orders.filter(
    clean_orders.customer_id.isNull()
).show()

+--------+-----------+-------+--------+-----+
|order_id|customer_id|product|quantity|price|
+--------+-----------+-------+--------+-----+
+--------+-----------+-------+--------+-----+



In [71]:
clean_orders.groupBy("order_id") \
    .count() \
    .filter("count > 1") \
    .show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [ ]:
##create a function 

In [72]:
def clean_order_data(df):

    df = df.dropna(
        subset=["customer_id"]
    )

    df = df.filter(
        df.quantity > 0
    )

    df = df.filter(
        df.price > 0
    )

    df = df.dropDuplicates(
        ["order_id"]
    )

    return df

In [73]:
clean_orders = clean_order_data(
    dirty_orders
)

In [74]:
clean_orders.show()

+--------+-----------+----------+--------+-----+
|order_id|customer_id|   product|quantity|price|
+--------+-----------+----------+--------+-----+
|    1001|       C001|    Laptop|       1|55000|
|    1002|       C002|     Mouse|       2|  800|
|    1003|       C001|  Keyboard|       1| 1500|
|    1004|       C003|   Monitor|       1|12000|
|    1005|       C002|Headphones|       1| 2500|
+--------+-----------+----------+--------+-----+



In [75]:
clean_orders = clean_orders.withColumn(
    "total_amount",
    clean_orders.quantity * clean_orders.price
)

In [76]:
clean_orders.show(0)

+--------+-----------+-------+--------+-----+------------+
|order_id|customer_id|product|quantity|price|total_amount|
+--------+-----------+-------+--------+-----+------------+
+--------+-----------+-------+--------+-----+------------+
only showing top 0 rows


In [77]:
clean_orders.show()

+--------+-----------+----------+--------+-----+------------+
|order_id|customer_id|   product|quantity|price|total_amount|
+--------+-----------+----------+--------+-----+------------+
|    1001|       C001|    Laptop|       1|55000|       55000|
|    1002|       C002|     Mouse|       2|  800|        1600|
|    1003|       C001|  Keyboard|       1| 1500|        1500|
|    1004|       C003|   Monitor|       1|12000|       12000|
|    1005|       C002|Headphones|       1| 2500|        2500|
+--------+-----------+----------+--------+-----+------------+



In [78]:
clean_orders.select(
    sum("total_amount").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|        72600|
+-------------+



In [79]:
customer_revenue = clean_orders.groupBy(
    "customer_id"
).agg(
    sum("total_amount").alias("...")
).orderBy(
    "...",
    ascending = False
)
customer_revenue.show()

AnalysisException: [INVALID_ATTRIBUTE_NAME_SYNTAX] Syntax error in the attribute name: .... Check that backticks appear in pairs, a quoted string is a complete name part and use a backtick only inside quoted name parts. SQLSTATE: 42601